<div style="width: 40%; clear: both;">
<div style="float: left; width: 20%;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f7/Universidad_T%C3%A9cnica_Patricular_de_Loja.png", align="left" width="280" height="120">
</div>
</div>
<div style="float: right; width: 70%;">
<p style="margin: 0; padding-top: 32px; text-align:right; color:#003366; font-size:16px"><u>Análisis de datos y visualización (CICO2000)</u></p>
<p style="margin: 0; text-align:right; color:#999999; font-size:17px">Redes y Analítica de Datos</p>
</div>
</div>
<div style="width: 100%; clear: both;">
<div style="width:100%;">&nbsp;</div>


# Notebook 1.1 - Tipos de datos y estructuras en Python

**Semana 3**: Apartados 1.8 y 1.9 de la guía didáctica.

En este notebook usted aprenderá a **crear, consultar y modificar** los tipos de datos de Python y las estructuras que más se usan en analítica, con ejemplos del dominio de *redes*.

**Objetivos**
1. Reconocer los tipos elementales `int`, `float`, `str` y `bool`, y convertir valores entre ellos.
2. Crear y manipular listas, diccionarios y conjuntos.
3. Crear y manipular `Series` y `DataFrame` de pandas.
4. Elegir la estructura adecuada según la información que quiere representar.

**Cómo usar este notebook**

- Inicie Google Colab o Jupyter Lab/Notebook y ejecute las celdas **en orden**, de arriba hacia abajo (`Shift + Enter`).
- Lea los comentarios (`# ...`) del código: explican qué hace cada línea.
- Las celdas **⚠️ Ponga atención** señalan errores frecuentes o conceptos que se evalúan.
- Las celdas **✍️ Pruebe usted** le piden modificar el código. Equivocarse forma parte del aprendizaje.
- La **sección 8** contiene la actividad práctica de la semana.

## 0. Preparación
Las estructuras de las secciones 1 a 4 son parte de Python y no necesitan instalación. Para las secciones 5 y 6 se usa la librería **pandas**, que se importa con el alias `pd` (convención usada en todo el mundo).

In [1]:
import pandas as pd          # librería para trabajar con datos tabulares

print("Versión de pandas:", pd.__version__)

Versión de pandas: 2.3.3


## 1. Tipos de datos elementales
Un **tipo de dato** indica qué clase de valor guarda una variable y qué operaciones se pueden hacer con él. En redes aparecen constantemente:

| Tipo | Qué representa | Ejemplos en redes |
|---|---|---|
| `int` | Números enteros | puerto 80 o 443, 1500 paquetes, 25 conexiones |
| `float` | Números con decimales | latencia 12.7 ms, pérdida 0.35 %, respuesta 1.9 s |
| `str` | Texto (cadenas) | IP `"192.168.1.10"`, host `"router-central"`, `"TCP"` |
| `bool` | Verdadero o falso | conectado `True`, alerta activa `False` |

En Python **no se declara el tipo**: se asigna a partir del valor. La función `type()` permite consultarlo.

In [53]:
# Enteros (int): cantidades que se cuentan
puerto = 443
paquetes = 1500
print(puerto, '-', paquetes)

# Decimales (float): medidas continuas. El separador decimal es el PUNTO, no la coma
latencia_ms = 12.7
perdida_pct = 0.35

# Texto (str): va entre comillas simples o dobles
ip = "192.168.1.10"
host = 'router-central'

# Lógicos (bool): solo True o False, con mayúscula inicial
conectado = True
alerta_activa = False

# type() devuelve el tipo de cada variable
print(puerto, type(puerto))
print(latencia_ms, type(latencia_ms))
print(ip, type(ip))
print(conectado, type(conectado))

443 - 1500
443 <class 'int'>
12.7 <class 'float'>
192.168.1.10 <class 'str'>
True <class 'bool'>


### 1.1 Operaciones con números

In [3]:
bytes_enviados = 2048
bytes_recibidos = 1500

total = bytes_enviados + bytes_recibidos     # suma
kb = total / 1024                            # la división / SIEMPRE devuelve float
bloques = total // 1024                      # división entera: descarta los decimales
resto = total % 1024                         # módulo: residuo de la división

print("Total de bytes:", total)
print("Total en KB:", kb)
print("Bloques completos de 1 KB:", bloques, "| bytes sobrantes:", resto)
print("KB redondeado a 2 decimales:", round(kb, 2))

Total de bytes: 3548
Total en KB: 3.46484375
Bloques completos de 1 KB: 3 | bytes sobrantes: 476
KB redondeado a 2 decimales: 3.46


### 1.2 Cadenas de texto (`str`)
Una cadena es una secuencia de caracteres. Tiene métodos útiles para limpiar y transformar texto, algo muy frecuente cuando los datos vienen de registros (*logs*) o de APIs.

In [4]:
protocolo = "  tcp "
estado = "Activo"

print(protocolo.strip().upper()) # quita espacios de los extremos y pasa a mayúsculas -> 'TCP'
print(estado.lower())            # minúsculas -> 'activo'
print(len(ip))                   # número de caracteres de la IP

# split() divide el texto en partes y devuelve una LISTA (sección 2)
octetos = ip.split(".")
print(octetos)

# f-strings: forma recomendada de combinar texto y variables
print(f"El host {host} ({ip}) responde en {latencia_ms} ms por el puerto {puerto}")

TCP
activo
12
['192', '168', '1', '10']
El host router-central (192.168.1.10) responde en 12.7 ms por el puerto 443


### 1.3 Booleanos y comparaciones
Las comparaciones (`==`, `!=`, `>`, `<`, `>=`, `<=`) devuelven un `bool`. Se combinan con `and`, `or` y `not`. Son la base de los **filtros** que usará en pandas.

In [5]:
umbral_ms = 20

latencia_alta = latencia_ms > umbral_ms       # False, porque 12.7 no supera 20
es_https = puerto == 443                      # == compara; = asigna

print("¿Latencia alta?", latencia_alta)
print("¿Es HTTPS?", es_https)
print("¿Conectado y sin alerta?", conectado and not alerta_activa)

¿Latencia alta? False
¿Es HTTPS? True
¿Conectado y sin alerta? True


### 1.4 Conversión entre tipos
Con frecuencia los datos llegan como texto (desde un CSV, un formulario o una API) y hay que convertirlos para poder calcular.

In [6]:
puerto_texto = "80"                   # parece número, pero es str

print(type(puerto_texto))
print(puerto_texto + puerto_texto)    # con texto, + CONCATENA: '8080'

puerto_num = int(puerto_texto)        # str -> int
print(puerto_num + puerto_num)        # ahora sí suma: 160

print(float("12.5"))                  # str -> float
print(str(443) + "/tcp")              # int -> str para armar texto
print(int(12.9))                      # float -> int TRUNCA (no redondea): 12

<class 'str'>
8080
160
12.5
443/tcp
12


> ⚠️ **Ponga atención**
> - `"80"` y `80` **no son lo mismo**: el primero es texto y el segundo un número. Con texto, `+` une cadenas en vez de sumar.
> - Una **dirección IP se guarda como `str`**: tiene varios puntos y no se hacen cálculos aritméticos con ella. Lo mismo ocurre con códigos como el número de cédula o un código postal.
> - Python distingue mayúsculas: `True` es un booleano, pero `true` provoca un error.
> - `int("12.5")` falla: primero convierta a `float` y luego a `int`.
> - `=` asigna un valor; `==` compara dos valores.

In [7]:
# Un error frecuente: convertir un texto que no representa un número.
# try/except captura el error para que el notebook siga ejecutándose.
try:
    int("cinco")
except ValueError as e:
    print("ValueError:", e)

ValueError: invalid literal for int() with base 10: 'cinco'


**✍️ Pruebe usted:** cree una variable `jitter_ms` con el valor `"3.8"` (texto), conviértala a `float` y muestre con un f-string si supera los 5 ms.

In [54]:
# Escriba su código aquí


## 2. Listas (`list`)
Una lista guarda **varios valores en orden**. Se crea con corchetes `[]` y es **mutable**: puede agregar, cambiar o quitar elementos después de crearla. Admite valores repetidos.

Se usa cuando importa el **orden** o cuando necesita recorrer una colección de valores: puertos a revisar, mediciones sucesivas, IPs observadas.

In [9]:
puertos = [80, 443, 22, 53]
protocolos = ["TCP", "UDP", "ICMP"]

print(puertos)
print(type(puertos))
print("Número de elementos:", len(puertos))

[80, 443, 22, 53]
<class 'list'>
Número de elementos: 4


### 2.1 Índices y segmentos (*slicing*)
Cada elemento ocupa una posición llamada **índice**.

| Elemento | 80 | 443 | 22 | 53 |
|---|---|---|---|---|
| Índice positivo | **0** | 1 | 2 | 3 |
| Índice negativo | -4 | -3 | -2 | **-1** |

In [10]:
print(puertos[0])      # primer elemento -> 80
print(puertos[1])      # segundo elemento -> 443
print(puertos[-1])     # último elemento -> 53

# Segmento [inicio:fin] -> el índice 'fin' NO se incluye
print(puertos[1:3])    # elementos de índice 1 y 2 -> [443, 22]
print(puertos[:2])     # los dos primeros
print(puertos[2:])     # desde el índice 2 hasta el final

80
443
53
[443, 22]
[80, 443]
[22, 53]


> ⚠️ **Ponga atención**
> - En Python los índices **empiezan en 0**: el **primer** elemento tiene índice 0 y el **segundo** tiene índice 1.
> - En un segmento `lista[a:b]` se incluye `a` pero **no** `b`.
> - Pedir un índice que no existe provoca un `IndexError`.

In [11]:
try:
    print(puertos[4])          # la lista tiene 4 elementos: índices 0 a 3
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


### 2.2 Modificar una lista

In [12]:
protocolos = ["TCP", "UDP", "ICMP"]

protocolos.append("HTTP")          # agrega al final
print("append:", protocolos)

protocolos[2] = "HTTPS"            # reemplaza el elemento de índice 2 (ICMP)
print("modificar:", protocolos)

protocolos.insert(0, "DNS")        # inserta en una posición concreta
print("insert:", protocolos)

protocolos.remove("UDP")           # elimina por VALOR (la primera coincidencia)
print("remove:", protocolos)

ultimo = protocolos.pop()          # elimina por POSICIÓN (por defecto, el último) y lo devuelve
print("pop:", protocolos, "| eliminado:", ultimo)

append: ['TCP', 'UDP', 'ICMP', 'HTTP']
modificar: ['TCP', 'UDP', 'HTTPS', 'HTTP']
insert: ['DNS', 'TCP', 'UDP', 'HTTPS', 'HTTP']
remove: ['DNS', 'TCP', 'HTTPS', 'HTTP']
pop: ['DNS', 'TCP', 'HTTPS'] | eliminado: HTTP


### 2.3 Recorrer, buscar y resumir

In [13]:
latencias = [12.5, 18.2, 10.7, 25.1]

# Recorrer con for: la variable 'lat' toma cada valor de la lista
for lat in latencias:
    estado = "ALTA" if lat > 20 else "normal"
    print(f"{lat:>5} ms -> {estado}")

# Pertenencia con 'in'
print("¿Se usa el puerto 22?", 22 in puertos)

# Funciones de resumen
print("Mínimo:", min(latencias), "| Máximo:", max(latencias))
print("Promedio:", round(sum(latencias) / len(latencias), 2))
print("Ordenadas:", sorted(latencias))       # sorted() no modifica la lista original

 12.5 ms -> normal
 18.2 ms -> normal
 10.7 ms -> normal
 25.1 ms -> ALTA
¿Se usa el puerto 22? True
Mínimo: 10.7 | Máximo: 25.1
Promedio: 16.62
Ordenadas: [10.7, 12.5, 18.2, 25.1]


> ⚠️ **Ponga atención:** con listas **no** se puede operar sobre todos los elementos a la vez. `latencias * 2` **repite** la lista; no multiplica cada valor. Para operar elemento por elemento se necesita un ciclo o una comprensión de lista. En la sección 5 verá que una `Series` de pandas sí lo hace directamente.

In [14]:
print(latencias * 2)                          # repite la lista: 8 elementos

latencias_seg = [lat / 1000 for lat in latencias]   # comprensión de lista: ms -> s
print(latencias_seg)

[12.5, 18.2, 10.7, 25.1, 12.5, 18.2, 10.7, 25.1]
[0.0125, 0.0182, 0.0107, 0.0251]


**✍️ Pruebe usted:** cree una lista `ips` con cuatro direcciones, agregue una quinta con `append`, muestre la segunda IP (¿qué índice usa?) y las dos últimas con un segmento.

In [55]:
# Escriba su código aquí


## 3. Diccionarios (`dict`)
Un diccionario guarda pares **clave: valor**. Se crea con llaves `{}`. En lugar de acceder por posición, se accede por **clave**, lo que lo hace ideal para describir un objeto con varios atributos: un dispositivo, una conexión, un usuario.

Es la estructura que más se parece al formato **JSON**, que es como responden la mayoría de **APIs** (tema 2.2 de la semana 4).

In [16]:
dispositivo = {
    "nombre": "Router-1",
    "ip": "192.168.1.1",
    "estado": "activo",
    "puertos_abiertos": [22, 80, 443]    # un valor puede ser una lista
}

print(dispositivo)
print(dispositivo["ip"])                  # acceso por clave
print(dispositivo["puertos_abiertos"][0]) # primero la clave, luego el índice de la lista

{'nombre': 'Router-1', 'ip': '192.168.1.1', 'estado': 'activo', 'puertos_abiertos': [22, 80, 443]}
192.168.1.1
22


### 3.1 Agregar, modificar y eliminar

In [17]:
dispositivo["ubicacion"] = "Laboratorio"   # si la clave NO existe, se agrega
dispositivo["estado"] = "inactivo"         # si la clave YA existe, se reemplaza el valor
del dispositivo["puertos_abiertos"]        # elimina el par clave-valor

print(dispositivo)
print("Número de claves:", len(dispositivo))

{'nombre': 'Router-1', 'ip': '192.168.1.1', 'estado': 'inactivo', 'ubicacion': 'Laboratorio'}
Número de claves: 4


### 3.2 Recorrer un diccionario

In [18]:
print("Claves:", list(dispositivo.keys()))
print("Valores:", list(dispositivo.values()))

# items() entrega cada par (clave, valor)
for clave, valor in dispositivo.items():
    print(f"{clave:>10}: {valor}")

Claves: ['nombre', 'ip', 'estado', 'ubicacion']
Valores: ['Router-1', '192.168.1.1', 'inactivo', 'Laboratorio']
    nombre: Router-1
        ip: 192.168.1.1
    estado: inactivo
 ubicacion: Laboratorio


> ⚠️ **Ponga atención**
> - Las claves son **únicas**: si repite una clave, el último valor reemplaza al anterior.
> - Acceder a una clave inexistente con `[]` provoca un `KeyError`. El método `.get()` devuelve `None` (o un valor por defecto) sin error.
> - Las claves distinguen mayúsculas: `"IP"` y `"ip"` son claves distintas.

In [19]:
try:
    print(dispositivo["mac"])
except KeyError as e:
    print("KeyError:", e)

print(dispositivo.get("mac"))                     # None, sin error
print(dispositivo.get("mac", "no registrada"))    # valor por defecto

KeyError: 'mac'
None
no registrada


### 3.3 Lista de diccionarios: registros como los de una API
Cuando una API devuelve varios dispositivos, normalmente entrega una **lista de diccionarios**. Cada diccionario es un registro (una fila) y cada clave es un campo (una columna).

In [20]:
dispositivos = [
    {"nombre": "Router-1", "ip": "192.168.1.1",  "latencia_ms": 12.5, "activo": True},
    {"nombre": "Switch-2", "ip": "192.168.1.2",  "latencia_ms": 18.2, "activo": True},
    {"nombre": "AP-3",     "ip": "192.168.1.20", "latencia_ms": 0.0,  "activo": False},
]

for d in dispositivos:
    if d["activo"]:
        print(f"{d['nombre']} responde en {d['latencia_ms']} ms")

Router-1 responde en 12.5 ms
Switch-2 responde en 18.2 ms


**✍️ Pruebe usted:** agregue a `dispositivos` un nuevo diccionario para un firewall activo y recorra la lista para mostrar solo los nombres de los equipos inactivos.

In [21]:
# Escriba su código aquí

## 4. Conjuntos (`set`)
Un conjunto guarda valores **únicos** y **sin orden**. Se crea con llaves `{}` (sin pares clave: valor) o con la función `set()`.

Es útil para **eliminar duplicados** (por ejemplo, obtener las IPs únicas de un registro de tráfico) y para comparar grupos.

In [22]:
protocolos_set = {"TCP", "UDP", "ICMP", "TCP"}    # el "TCP" repetido se descarta
print(protocolos_set)
print(len(protocolos_set))

# Caso típico: IPs únicas a partir de una lista con repeticiones
ips = ["192.168.1.1", "192.168.1.1", "10.0.0.2"]
ips_unicas = set(ips)
print("IPs únicas:", ips_unicas, "| total:", len(ips_unicas))

{'TCP', 'UDP', 'ICMP'}
3
IPs únicas: {'192.168.1.1', '10.0.0.2'} | total: 2


### 4.1 Modificar y consultar

In [23]:
protocolos_set.add("HTTP")          # agrega (si ya existe, no hace nada)
protocolos_set.remove("UDP")        # elimina; da error si el valor no existe
protocolos_set.discard("SMTP")      # elimina sin error aunque no exista

print(protocolos_set)
print("¿Se usa TCP?", "TCP" in protocolos_set)   # búsqueda muy rápida

{'TCP', 'ICMP', 'HTTP'}
¿Se usa TCP? True


### 4.2 Operaciones entre conjuntos

In [24]:
ips_lunes = {"10.0.0.5", "192.168.1.10", "192.168.1.15"}
ips_martes = {"192.168.1.10", "10.0.0.8"}

print("Unión (vistas algún día):", ips_lunes | ips_martes)
print("Intersección (ambos días):", ips_lunes & ips_martes)
print("Diferencia (solo el lunes):", ips_lunes - ips_martes)

Unión (vistas algún día): {'10.0.0.5', '192.168.1.10', '192.168.1.15', '10.0.0.8'}
Intersección (ambos días): {'192.168.1.10'}
Diferencia (solo el lunes): {'10.0.0.5', '192.168.1.15'}


> ⚠️ **Ponga atención**
> - `{}` vacío crea un **diccionario**, no un conjunto. Un conjunto vacío se crea con `set()`.
> - Un conjunto **no tiene índices**: `protocolos_set[0]` provoca un `TypeError`. Si necesita orden, conviértalo con `sorted()` o `list()`.
> - El orden en que se muestran los elementos puede variar; no dependa de él.

In [25]:
print(type({}), type(set()))

try:
    protocolos_set[0]
except TypeError as e:
    print("TypeError:", e)

print(sorted(protocolos_set))     # lista ordenada alfabéticamente

<class 'dict'> <class 'set'>
TypeError: 'set' object is not subscriptable
['HTTP', 'ICMP', 'TCP']


## 5. Series de pandas
Una **Series** es una columna de datos con un **índice** (etiquetas para cada valor) y, opcionalmente, un **nombre**. Es como una lista con superpoderes: permite calcular estadísticas, filtrar y operar sobre todos los valores a la vez.

In [26]:
latencias = pd.Series([12.5, 18.2, 10.7, 25.1], name="latencia_ms")
print(latencias)

0    12.5
1    18.2
2    10.7
3    25.1
Name: latencia_ms, dtype: float64


La columna de la izquierda (0, 1, 2, 3) es el **índice**; abajo aparecen el nombre y el tipo (`dtype`). El índice puede tener etiquetas propias:

In [27]:
latencias_host = pd.Series([12.5, 18.2, 10.7, 25.1],
                           index=["router-1", "switch-2", "ap-3", "firewall"],
                           name="latencia_ms")
print(latencias_host)
print()
print("Por etiqueta:", latencias_host["ap-3"])
print("Por posición:", latencias_host.iloc[0])     # iloc = posición entera

router-1    12.5
switch-2    18.2
ap-3        10.7
firewall    25.1
Name: latencia_ms, dtype: float64

Por etiqueta: 10.7
Por posición: 12.5


### 5.1 Estadísticas, filtros y modificación

In [28]:
print("Promedio:", latencias.mean())
print("Máximo:", latencias.max(), "| Mínimo:", latencias.min())

# Filtro: la comparación devuelve una Series de booleanos...
print(latencias > 15)

# ...que se usa entre corchetes para quedarse solo con los True
print(latencias[latencias > 15])

Promedio: 16.625
Máximo: 25.1 | Mínimo: 10.7
0    False
1     True
2    False
3     True
Name: latencia_ms, dtype: bool
1    18.2
3    25.1
Name: latencia_ms, dtype: float64


In [29]:
latencias[1] = 17.5          # modifica el valor con índice 1
print(latencias)

# Operación vectorizada: se aplica a TODOS los valores sin escribir un ciclo
print(latencias / 1000)      # ms -> s

0    12.5
1    17.5
2    10.7
3    25.1
Name: latencia_ms, dtype: float64
0    0.0125
1    0.0175
2    0.0107
3    0.0251
Name: latencia_ms, dtype: float64


> ⚠️ **Ponga atención**
> - Compare `latencias / 1000` (Series) con la lista de la sección 2: pandas opera elemento por elemento de forma automática.
> - El filtro se escribe `serie[serie > valor]`; la parte interna produce valores `True`/`False`.
> - `latencias[1]` busca la **etiqueta** 1 del índice. Si el índice tiene texto (como en `latencias_host`), use `.iloc[pos]` para acceder por posición.

**✍️ Pruebe usted:** cree una Series `perdida` con los valores 0.1, 0.0, 2.3, 0.4 y nombre `perdida_pct`. Calcule su promedio y muestre solo los valores mayores que 0.

In [30]:
# Escriba su código aquí



## 6. DataFrame de pandas
Un **DataFrame** es una **tabla**: filas (registros) y columnas (variables). **Cada columna es una Series** y todas comparten el mismo índice. Es la estructura central del análisis de datos en Python.

Una forma común de crearlo es con un diccionario: cada clave es el nombre de una columna y cada valor, una lista con los datos de esa columna.

In [31]:
trafico_red = pd.DataFrame({
    "ip_origen":  ["192.168.1.10", "192.168.1.15", "10.0.0.5"],
    "ip_destino": ["10.0.0.20", "10.0.0.25", "192.168.1.1"],
    "protocolo":  ["TCP", "UDP", "TCP"],
    "puerto":     [443, 53, 80],
    "bytes":      [1500, 512, 2048]
}, index=["Registro 1", "Registro 2", "Registro 3"])

trafico_red

,ip_origen,ip_destino,protocolo,puerto,bytes
Registro 1,192.168.1.10,10.0.0.20,TCP,443,1500
Registro 2,192.168.1.15,10.0.0.25,UDP,53,512
Registro 3,10.0.0.5,192.168.1.1,TCP,80,2048


> ⚠️ **Ponga atención:** todas las listas deben tener **la misma longitud** (una por fila). Si una lista tiene un elemento menos, pandas lanza un `ValueError`.

### 6.1 Explorar la estructura

In [32]:
print("Dimensiones (filas, columnas):", trafico_red.shape)
print("Columnas:", list(trafico_red.columns))
print("Índice:", list(trafico_red.index))
print()
print(trafico_red.dtypes)

Dimensiones (filas, columnas): (3, 5)
Columnas: ['ip_origen', 'ip_destino', 'protocolo', 'puerto', 'bytes']
Índice: ['Registro 1', 'Registro 2', 'Registro 3']

ip_origen     object
ip_destino    object
protocolo     object
puerto         int64
bytes          int64
dtype: object


> ⚠️ **Ponga atención**
> - `shape`, `columns`, `index` y `dtypes` son **atributos**: se escriben **sin paréntesis**. `trafico_red.shape()` produce un error.
> - `head()`, `info()` y `describe()` son **métodos**: llevan paréntesis.
> - Las columnas de texto aparecen como `object` (pandas 2, versión habitual en Colab) o como `str` (pandas 3). En ambos casos significa texto.

In [33]:
trafico_red.info()          # resumen: filas, columnas, tipos y valores no nulos

<class 'pandas.core.frame.DataFrame'>
Index: 3 entries, Registro 1 to Registro 3
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ip_origen   3 non-null      object
 1   ip_destino  3 non-null      object
 2   protocolo   3 non-null      object
 3   puerto      3 non-null      int64 
 4   bytes       3 non-null      int64 
dtypes: int64(2), object(3)
memory usage: 144.0+ bytes


In [34]:
trafico_red.describe()      # estadísticas de las columnas numéricas

,puerto,bytes
count,3.000000,3.000000
mean,192.000000,1353.333333
std,217.791184,778.432613
min,53.000000,512.000000
25%,66.500000,1006.000000
50%,80.000000,1500.000000
75%,261.500000,1774.000000
max,443.000000,2048.000000


In [35]:
trafico_red.head(2)         # primeras n filas (por defecto 5)

,ip_origen,ip_destino,protocolo,puerto,bytes
Registro 1,192.168.1.10,10.0.0.20,TCP,443,1500
Registro 2,192.168.1.15,10.0.0.25,UDP,53,512


### 6.2 Seleccionar columnas

In [36]:
# Una columna (corchetes simples) -> devuelve una Series
print(type(trafico_red["protocolo"]))
trafico_red["protocolo"]

<class 'pandas.core.series.Series'>


Registro 1    TCP
Registro 2    UDP
Registro 3    TCP
Name: protocolo, dtype: object

In [37]:
# Varias columnas (DOBLE corchete: una lista de nombres) -> devuelve un DataFrame
trafico_red[["ip_origen", "protocolo"]]

,ip_origen,protocolo
Registro 1,192.168.1.10,TCP
Registro 2,192.168.1.15,UDP
Registro 3,10.0.0.5,TCP


### 6.3 Seleccionar filas: `loc` e `iloc`
- `loc` selecciona por **etiqueta** del índice (y nombre de columna).
- `iloc` selecciona por **posición** entera, empezando en 0.

In [38]:
print(trafico_red.loc["Registro 2"])             # una fila por etiqueta
print()
print(trafico_red.iloc[0])                        # primera fila por posición
print()
print(trafico_red.loc["Registro 3", "bytes"])     # una celda: fila y columna

ip_origen     192.168.1.15
ip_destino       10.0.0.25
protocolo              UDP
puerto                  53
bytes                  512
Name: Registro 2, dtype: object

ip_origen     192.168.1.10
ip_destino       10.0.0.20
protocolo              TCP
puerto                 443
bytes                 1500
Name: Registro 1, dtype: object

2048


In [39]:
trafico_red.iloc[0:2, [0, 2]]                     # filas 0-1 y columnas 0 y 2

,ip_origen,protocolo
Registro 1,192.168.1.10,TCP
Registro 2,192.168.1.15,UDP


### 6.4 Filtrar filas
Igual que con las Series, se escribe una condición que produce `True`/`False` y se coloca entre corchetes. Para combinar condiciones se usan `&` (y) y `|` (o), con **cada condición entre paréntesis**.

In [40]:
trafico_red[trafico_red["protocolo"] == "TCP"]

,ip_origen,ip_destino,protocolo,puerto,bytes
Registro 1,192.168.1.10,10.0.0.20,TCP,443,1500
Registro 3,10.0.0.5,192.168.1.1,TCP,80,2048


In [41]:
trafico_red[(trafico_red["protocolo"] == "TCP") & (trafico_red["bytes"] > 1600)]

,ip_origen,ip_destino,protocolo,puerto,bytes
Registro 3,10.0.0.5,192.168.1.1,TCP,80,2048


### 6.5 Crear y modificar columnas, y calcular estadísticas

In [42]:
trafico_red["trafico_kb"] = trafico_red["bytes"] / 1024       # nueva columna calculada
trafico_red["seguro"] = trafico_red["puerto"] == 443          # columna booleana
trafico_red.loc["Registro 2", "bytes"] = 600                  # modificar una celda

trafico_red

,ip_origen,ip_destino,protocolo,puerto,bytes,trafico_kb,seguro
Registro 1,192.168.1.10,10.0.0.20,TCP,443,1500,1.464844,True
Registro 2,192.168.1.15,10.0.0.25,UDP,53,600,0.500000,False
Registro 3,10.0.0.5,192.168.1.1,TCP,80,2048,2.000000,False


In [43]:
print("Promedio de bytes:", trafico_red["bytes"].mean())
print("Máximo de KB:", round(trafico_red["trafico_kb"].max(), 2))
print()
print(trafico_red["protocolo"].value_counts())   # frecuencia de cada protocolo

Promedio de bytes: 1382.6666666666667
Máximo de KB: 2.0

protocolo
TCP    2
UDP    1
Name: count, dtype: int64


> ⚠️ **Ponga atención**
> - `df["col"]` → Series; `df[["col"]]` o `df[["a", "b"]]` → DataFrame.
> - En filtros con varias condiciones use `&` y `|`, **no** `and` / `or`, y encierre cada condición entre paréntesis.
> - Observe que, tras modificar `bytes` del Registro 2, la columna `trafico_kb` **no se actualiza sola**: fue calculada antes. Si cambia los datos de origen, vuelva a calcular las columnas derivadas.

### 6.6 De una lista de diccionarios a un DataFrame
Esto conecta las secciones 3 y 6: una respuesta JSON de una API (lista de diccionarios) se convierte en tabla con una sola línea.

In [44]:
df_dispositivos = pd.DataFrame(dispositivos)
df_dispositivos

,nombre,ip,latencia_ms,activo
0,Router-1,192.168.1.1,12.5,True
1,Switch-2,192.168.1.2,18.2,True
2,AP-3,192.168.1.20,0.0,False


**✍️ Pruebe usted:** a partir de `trafico_red`, (a) muestre solo las filas con protocolo `UDP`; (b) cree la columna `paquetes` estimando 1 paquete por cada 500 bytes (división entera `//`); (c) muestre las columnas `ip_origen` y `paquetes`.

In [45]:
# Escriba su código aquí

## 7. Integración
En un caso real se combinan varias estructuras. Observe qué estructura se usa para cada tipo de información.

In [46]:
# Lista: registros en orden de llegada (pueden repetirse)
protocolos_obs = ["TCP", "UDP", "TCP", "ICMP"]

# Conjunto: valores únicos de esa lista
print("Protocolos distintos:", set(protocolos_obs))

# Diccionario: atributos de UN dispositivo
router = {"nombre": "Router-1", "ip": "192.168.1.1", "estado": "activo"}
print("Dispositivo:", router["nombre"], "-", router["estado"])

# Series: una variable numérica sobre la que se calculan estadísticas
latencias = pd.Series([12.5, 18.2, 10.7, 25.1], name="latencia_ms")
print("Latencia promedio:", latencias.mean(), "ms")

# DataFrame: varias variables por registro (tabla)
print(trafico_red[["protocolo", "bytes"]])

Protocolos distintos: {'TCP', 'UDP', 'ICMP'}
Dispositivo: Router-1 - activo
Latencia promedio: 16.625 ms
           protocolo  bytes
Registro 1       TCP   1500
Registro 2       UDP    600
Registro 3       TCP   2048


### 7.1 ¿Qué estructura elegir?
Antes de escribir código, hágase estas preguntas:

| Pregunta | Si la respuesta es sí… |
|---|---|
| ¿Es un solo valor (un puerto, una latencia, una IP)? | tipo elemental (`int`, `float`, `str`, `bool`) |
| ¿Necesito conservar el orden o modificar la colección? | `list` |
| ¿Describo un objeto con atributos con nombre? | `dict` |
| ¿Necesito eliminar duplicados o comparar grupos? | `set` |
| ¿Hago cálculos y filtros sobre una sola variable? | `pd.Series` |
| ¿Trabajo con varias variables por registro (tabla)? | `pd.DataFrame` |

| Estructura | Ordenada | Mutable | Duplicados | Acceso |
|---|---|---|---|---|
| `list` | Sí | Sí | Sí | por posición `[0]` |
| `dict` | Sí (orden de inserción) | Sí | Claves únicas | por clave `["ip"]` |
| `set` | No | Sí | No | solo `in` / recorrido |
| `Series` | Sí | Sí | Sí | por etiqueta o posición (`.iloc`) |
| `DataFrame` | Sí | Sí | Sí | `["col"]`, `.loc`, `.iloc` |

## 8. Actividad práctica
Use los siguientes datos. Resuelva cada literal en la celda indicada y **agregue comentarios** que expliquen su código.

| Información | Datos |
|---|---|
| Protocolos | TCP, UDP, ICMP |
| Puertos | 80, 443, 53, 22 |
| Dispositivo | Router-1, IP 192.168.1.1, estado activo |
| Latencias (ms) | 12.5, 18.2, 10.7, 25.1 |
| IPs observadas | 192.168.1.10, 192.168.1.15, 192.168.1.10, 10.0.0.5 |

**a)** Cree una **lista** con los protocolos, agregue `"HTTP"` y muestre el primer y el último elemento.

In [47]:
# a) Escriba su código aquí



**b)** Cree una **lista** con los puertos, cambie el puerto 22 por 8080 y verifique si el puerto 53 está en la lista.

In [48]:
# b) Escriba su código aquí

**c)** Cree un **diccionario** para el dispositivo, agregue la clave `"ubicacion"`, cambie el estado a `"inactivo"` y recorra sus pares clave-valor.

In [49]:
# c) Escriba su código aquí

**d)** Con las IPs observadas, obtenga las **IPs únicas** usando un conjunto e indique cuántas son.

In [50]:
# d) Escriba su código aquí

**e)** Cree una **Series** con las latencias (nombre `latencia_ms`), calcule el promedio, el máximo y el mínimo, y muestre las latencias mayores a 15 ms.

In [51]:
# e) Escriba su código aquí

**f)** Cree un **DataFrame** con cuatro registros y las columnas `ip_origen`, `protocolo`, `puerto` y `latencia_ms`. Muestre su `shape`, filtre los registros TCP y agregue una columna `latencia_s` (latencia en segundos).

In [52]:
# f) Escriba su código aquí

**Reflexión:** ¿qué estructura resultó más adecuada para cada tipo de información y por qué? Responda en esta celda (doble clic para editar).

*Su respuesta:*

## Resumen
- Los tipos elementales (`int`, `float`, `str`, `bool`) representan valores individuales; verifique siempre el tipo con `type()` y convierta cuando sea necesario.
- `list` → colección ordenada y modificable; `dict` → atributos con nombre (como JSON); `set` → valores únicos.
- `Series` → una variable con índice y operaciones vectorizadas; `DataFrame` → tabla donde cada columna es una Series.
- Elegir bien la estructura simplifica el análisis: depende de si necesita orden, nombres, unicidad, cálculos o una tabla.